# 06 — Model Training: PCOS Clinical Dataset

**Goal:** Train multiple classifiers to predict **PCOS / No PCOS** (binary) from clinical lab data.

| Step | What | Why |
|------|------|-----|
| 1 | Load processed data | Already scaled + derived features from FE notebook |
| 2 | Train 5 models | LR, RF, XGBoost |
| 3 | Evaluate with 4 metrics | Accuracy, F1 (Harmonic mean of precision and recall), ROC-AUC, Confusion Matrix |
| 4 | Cross-validate top models | Guard against split variance |
| 5 | Save all trained models | Evaluation notebook loads these |

In [20]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    roc_auc_score,
    confusion_matrix, 
    classification_report
)
from sklearn.model_selection import cross_val_score, StratifiedKFold , GridSearchCV
from xgboost import XGBClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline          import Pipeline

os.makedirs("../models/clinical", exist_ok=True)

print("Libraries loaded ✓")

Libraries loaded ✓


## 1. Load Processed Data

In [21]:
X_train = pd.read_csv("../data/processed/clinical_X_train.csv")
y_train = pd.read_csv("../data/processed/clinical_y_train.csv").squeeze()
X_test  = pd.read_csv("../data/processed/clinical_X_test.csv")
y_test  = pd.read_csv("../data/processed/clinical_y_test.csv").squeeze()

label_names = ["No PCOS", "PCOS"]

print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"Train class balance: {dict(y_train.value_counts())}")
print(f"Test class balance : {dict(y_test.value_counts())}")

X_train shape : (374, 54)
X_test shape  : (94, 54)
Train class balance: {0: np.int64(208), 1: np.int64(166)}
Test class balance : {0: np.int64(52), 1: np.int64(42)}


## 2. Define Models

| Model | Why include it |
|-------|---------------|
| Logistic Regression | Fast linear baseline; interpretable coefficients |
| Random Forest | Handles non-linearity; built-in feature importance |
| XGBoost | Best gradient boosting; usually top performer |

Clinical data is **already scaled** (StandardScaler fitted in FE notebook), so all models receive the same input.

In [22]:
# =====================================================
# SETTINGS
# =====================================================

SEED = 42

SCALE_POS_WEIGHT = (
    (y_train == 0).sum() / (y_train == 1).sum()
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

# =====================================================
# PIPELINES + PARAM GRIDS
# =====================================================

PIPELINES = {

    "Logistic Regression": (
        Pipeline([
            ("sel", SelectKBest(f_classif)),
            ("clf", LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=SEED
            ))
        ]),

        {
            "sel__k": [15, 20, 25, "all"],

            "clf__C": [0.01, 0.1, 1, 10],

            "clf__solver": ["lbfgs", "saga"],

            "clf__penalty": ["l2"],
        }
    ),

    "Random Forest": (
        Pipeline([
            ("sel", SelectKBest(f_classif)),
            ("clf", RandomForestClassifier(
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1
            ))
        ]),

        {
            "sel__k": [15, 20, "all"],

            "clf__n_estimators": [200, 400],

            "clf__max_depth": [None, 6, 10],

            "clf__min_samples_split": [2, 5],

            "clf__max_features": ["sqrt", 0.5],
        }
    ),

    "XGBoost": (
        Pipeline([
            ("sel", SelectKBest(f_classif)),
            ("clf", XGBClassifier(
                scale_pos_weight=SCALE_POS_WEIGHT,
                eval_metric="logloss",
                use_label_encoder=False,
                random_state=SEED,
                n_jobs=-1
            ))
        ]),

        {
            "sel__k": [15, 20, "all"],

            "clf__n_estimators": [100, 200, 300],

            "clf__max_depth": [3, 5, 7],

            "clf__learning_rate": [0.05, 0.1, 0.2],

            "clf__subsample": [0.7, 1.0],

            "clf__colsample_bytree": [0.7, 1.0],
        }
    ),
}

print(f"Models to train: {list(PIPELINES.keys())}")

Models to train: ['Logistic Regression', 'Random Forest', 'XGBoost']


## 3. Train & Evaluate All Models

In [24]:
# =====================================================
# TRAINING + GRID SEARCH
# =====================================================

results = {}
summary_rows = []

for name, (pipeline, param_grid) in PIPELINES.items():

    print(f"\n{'─'*60}")
    print(f" Tuning: {name}")
    print(f"{'─'*60}")

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1,
        refit=True,
        verbose=1
    )

    # Train
    grid.fit(X_train, y_train)

    # Best model
    best_model = grid.best_estimator_

    print("\n Best Parameters:")
    print(grid.best_params_)

    print(f"\n Best CV ROC-AUC: {grid.best_score_:.4f}")

    # =====================================================
    # PREDICTIONS
    # =====================================================

    y_pred = best_model.predict(X_test)

    y_proba = best_model.predict_proba(X_test)[:, 1]

    # =====================================================
    # METRICS
    # =====================================================

    acc = accuracy_score(y_test, y_pred)

    f1_pcos = f1_score(
        y_test,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    f1_macro = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    roc = roc_auc_score(y_test, y_proba)

    cm = confusion_matrix(y_test, y_pred)

    # =====================================================
    # SAVE RESULTS
    # =====================================================

    results[name] = {
        "best_model": best_model,
        "best_params": grid.best_params_,
        "cv_roc_auc": round(grid.best_score_, 4),
        "accuracy": round(acc, 4),
        "f1_pcos": round(f1_pcos, 4),
        "f1_macro": round(f1_macro, 4),
        "roc_auc": round(roc, 4),
        "cm": cm,
        "y_proba": y_proba
    }

    summary_rows.append({
        "Model": name,
        "CV ROC-AUC": round(grid.best_score_, 4),
        "Test Accuracy": round(acc, 4),
        "F1 PCOS": round(f1_pcos, 4),
        "F1 Macro": round(f1_macro, 4),
        "Test ROC-AUC": round(roc, 4)
    })

    # =====================================================
    # PRINT RESULTS
    # =====================================================

    print(f"\n Accuracy        : {acc:.4f}")
    print(f" F1 (PCOS)       : {f1_pcos:.4f}")
    print(f" F1 Macro        : {f1_macro:.4f}")
    print(f" Test ROC-AUC    : {roc:.4f}")

    print("\n Confusion Matrix:")

    print(
        pd.DataFrame(
            cm,
            index=label_names,
            columns=label_names
        ).to_string()
    )

    print("\n Classification Report:")

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=label_names,
            zero_division=0
        )
    )

# =====================================================
# FINAL COMPARISON TABLE
# =====================================================

summary_df = pd.DataFrame(summary_rows)

print("\n")
print("="*75)
print(" FINAL MODEL COMPARISON")
print("="*75)

print(
    summary_df.sort_values(
        by="F1 Macro",
        ascending=False
    ).to_string(index=False)
)


────────────────────────────────────────────────────────────
 Tuning: Logistic Regression
────────────────────────────────────────────────────────────
Fitting 5 folds for each of 32 candidates, totalling 160 fits

 Best Parameters:
{'clf__C': 0.01, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs', 'sel__k': 20}

 Best CV ROC-AUC: 0.5092

 Accuracy        : 0.3617
 F1 (PCOS)       : 0.3617
 F1 Macro        : 0.3617
 Test ROC-AUC    : 0.3443

 Confusion Matrix:
         No PCOS  PCOS
No PCOS       17    35
PCOS          25    17

 Classification Report:
              precision    recall  f1-score   support

     No PCOS       0.40      0.33      0.36        52
        PCOS       0.33      0.40      0.36        42

    accuracy                           0.36        94
   macro avg       0.37      0.37      0.36        94
weighted avg       0.37      0.36      0.36        94


────────────────────────────────────────────────────────────
 Tuning: Random Forest
─────────────────────────────────

## 4. Summary Table

In [25]:
summary = pd.DataFrame([
    {
        "Model":       name,
        "Accuracy":    r["accuracy"],
        "F1 (PCOS)":   r["f1_pcos"],
        "F1 Macro":    r["f1_macro"],
        "ROC-AUC":     r["roc_auc"],
    }
    for name, r in results.items()
]).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)

print("\n=== CLINICAL — All Models Summary (sorted by ROC-AUC) ===")
print(summary.to_string(index=False))
print(f"\n→ Best model: {summary.iloc[0]['Model']}")


=== CLINICAL — All Models Summary (sorted by ROC-AUC) ===
              Model  Accuracy  F1 (PCOS)  F1 Macro  ROC-AUC
            XGBoost    0.5638     0.4810    0.5524   0.5659
      Random Forest    0.5000     0.3562    0.4737   0.5101
Logistic Regression    0.3617     0.3617    0.3617   0.3443

→ Best model: XGBoost


## 5. Cross-Validation on Top-2 Models

In [27]:

X_all = pd.concat([X_train, X_test], ignore_index=True)
y_all = pd.concat([y_train, y_test], ignore_index=True)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

top2 = summary["Model"].values[:2]

print("\n5-Fold Cross-Validation (ROC-AUC) — Top 2 Models:")

for name in top2:

    # IMPORTANT: use best_model (GridSearchCV result)
    model = results[name]["best_model"]

    scores = cross_val_score(
        model,
        X_all,
        y_all,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    print(
        f"  {name:<22} "
        f"mean={scores.mean():.4f}  "
        f"std={scores.std():.4f}  "
        f"folds={np.round(scores, 4)}"
    )


5-Fold Cross-Validation (ROC-AUC) — Top 2 Models:
  XGBoost                mean=0.5438  std=0.0367  folds=[0.5174 0.6081 0.5014 0.5385 0.5535]
  Random Forest          mean=0.5300  std=0.0375  folds=[0.4712 0.5829 0.5394 0.5098 0.5469]


## 6. Save All Trained Models

In [29]:
for name, r in results.items():
    safe_name = name.lower().replace(" ", "_")
    path = f"../models/clinical/{safe_name}.pkl"
    joblib.dump(r["best_model"], path)
    print(f"  Saved: {path}")

# Save summary for evaluation notebook
summary.to_csv("../models/clinical/training_summary.csv", index=False)

print("\nAll models saved ✓")
print(f"Best model: {summary.iloc[0]['Model']} (ROC-AUC={summary.iloc[0]['ROC-AUC']})")

  Saved: ../models/clinical/logistic_regression.pkl
  Saved: ../models/clinical/random_forest.pkl
  Saved: ../models/clinical/xgboost.pkl

All models saved ✓
Best model: XGBoost (ROC-AUC=0.5659)


## Summary

```
Task    : Binary Classification — PCOS / No PCOS
Dataset : Clinical — StandardScaler + derived features (FAI, follicle totals, ovary volumes)
Models  : Logistic Regression, Random Forest, XGBoost
Metrics : Accuracy, F1 (PCOS class), F1 Macro, ROC-AUC

Saved artifacts:
  ../models/clinical/logistic_regression.pkl
  ../models/clinical/random_forest.pkl
  ../models/clinical/xgboost.pkl
  ../models/clinical/training_summary.csv

```